# 101 — SWE-Agent Code Editing
## Build an autonomous bug-fixing agent with the ACI pattern
⏱ ~45 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/101-swe-agent-code-editing/swe_agent_code_editing_workbook.ipynb)

Software Engineering (SWE) agents are LLM-based systems that can read code, locate bugs, apply fixes, and verify correctness — all without human guidance. This example implements the core idea from **Yang et al. 2024 (SWE-agent)**: give a language model a minimal but well-designed set of file-editing tools and a failing test suite, then let it iterate until every test passes.

The pattern you'll build here — view → edit → test → repeat — is the same loop that powers production coding assistants. Understanding it from first principles makes you a better designer of agentic systems.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — SWE-agent, ACI, ReAct loop |
| 2 | **Setup** — install deps, load API key |
| 3 | **The Bugs** — write a buggy file and failing tests |
| 4 | **The Tools** — view_file, edit_file, run_tests |
| 5 | **The Workflow** — ReAct agent with `create_react_agent` |
| 6 | **Run It** — watch the agent debug live |
| 7 | **Trace the Loop** — inspect message history |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab (free tier works)
- `OPENAI_API_KEY` in a `.env` file (local) or Colab Secrets
- Familiarity with Python functions and basic pytest

### Key References
> Yang et al. 2024 — *SWE-agent: Agent-Computer Interfaces Enable Automated Software Engineering* — [arxiv.org/abs/2405.15793](https://arxiv.org/abs/2405.15793)
>
> LangGraph `create_react_agent` — [langchain-ai.github.io/langgraph/reference/prebuilt](https://langchain-ai.github.io/langgraph/reference/prebuilt/)
>
> LangChain `@tool` decorator — [python.langchain.com/docs/concepts/tools](https://python.langchain.com/docs/concepts/tools/)

## Part 1 — Concepts

### What is a SWE-agent?

A **SWE-agent** is an LLM equipped with file-system tools so it can act like a developer: read source files, understand failing tests, make targeted edits, and verify fixes. The key insight from Yang et al. 2024 is that the *interface design* — not just the model — determines success.

### The ACI Pattern (Agent-Computer Interface)

The ACI is the minimal toolset exposed to the agent. Good ACIs are:
- **Minimal** — only what the agent actually needs
- **Lossless** — the agent gets the full information it needs per call
- **Actionable** — outputs are directly interpretable without post-processing

For code repair, three tools are sufficient:

```
┌──────────────────────────────────────────────┐
│              AGENT-COMPUTER INTERFACE         │
│                                              │
│  view_file(path)          → file contents    │
│  edit_file(path, old, new)→ targeted replace │
│  run_tests(workspace)     → pass/fail output │
└──────────────────────────────────────────────┘
```

### The ReAct Loop

The agent runs in a **ReAct** (Reason + Act) loop — it alternates between reasoning about what to do and taking an action (calling a tool). `create_react_agent` from LangGraph compiles this automatically:

```
 ┌──────────────────────────────────────────────────────────┐
 │                      ReAct Loop                          │
 │                                                          │
 │   Task ──► LLM reasons ──► tool call ──► tool result     │
 │               ▲                              │           │
 │               └──────────────────────────────┘           │
 │                                                          │
 │   Loop exits when LLM emits a response with NO tool call │
 └──────────────────────────────────────────────────────────┘
```

### Why This Matters

The SWE-agent paper showed that GPT-4 with a good ACI could resolve **12.5% of real GitHub issues** — without any fine-tuning. The bottleneck was the interface, not the model. Understanding this helps you:
- Design better tool interfaces for your own agents
- Know when to add vs. remove tools
- Reason about agent loop termination

## Part 2 — Setup

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "langgraph", "langchain-openai", "langchain-core", "python-dotenv"],
        check=True
    )
    print("Colab install complete.")
else:
    print("Local — skipping install. Ensure requirements.txt deps are installed.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
if not (bool(key) and key.startswith('sk-')):
    raise RuntimeError("OPENAI_API_KEY is required for the live SWE-agent cells.")
print("API key ready: True")

## Part 3 — The Bugs

The agent's task is to fix a real (deliberately broken) Python file. We write both files to a temporary directory at runtime — this mimics the agent working inside a real repository.

### Bug 1: `find_max` — wrong initial value

```python
def find_max(nums):
    max_val = 0  # BUG: should be nums[0]
    ...
```

If all values are negative (e.g. `[-1, -2, -3]`), the loop never updates `max_val` because every element is less than `0`. The function returns `0` instead of `-1`.

**Fix**: initialise `max_val = nums[0]`

### Bug 2: `average` — spurious arithmetic

```python
def average(nums):
    return sum(nums) / len(nums) - 1  # BUG: spurious -1
```

`average([1, 2, 3, 4, 5])` returns `2.0` instead of `3.0` because of the trailing `- 1`.

**Fix**: remove the `- 1`

The agent must discover both bugs by reading test output alone — exactly as a human developer would.

In [ ]:
# The buggy source code — identical to src/tools.py BUGGY_CODE constant
BUGGY_CODE = """\
def find_max(nums):
    max_val = 0  # bug: should be nums[0]
    for n in nums:
        if n > max_val:
            max_val = n
    return max_val

def average(nums):
    return sum(nums) / len(nums) - 1  # bug: spurious -1
"""

# The test file that exercises both buggy functions
TEST_CODE = """\
from buggy import find_max, average

def test_find_max():
    assert find_max([3, 1, 4, 1, 5, 9]) == 9
    assert find_max([-1, -2, -3]) == -1   # fails with max_val=0

def test_average():
    assert average([1, 2, 3, 4, 5]) == 3.0
    assert average([10, 20]) == 15.0       # fails due to -1

if __name__ == "__main__":
    test_find_max()
    test_average()
    print("All tests passed!")
"""

print("BUGGY_CODE:")
print(BUGGY_CODE)
print("TEST_CODE:")
print(TEST_CODE)

In [ ]:
import os
import tempfile

def setup_workspace() -> str:
    """Write buggy.py and test_buggy.py to a fresh temp directory."""
    d = tempfile.mkdtemp(prefix="swe_")
    open(os.path.join(d, "buggy.py"), "w").write(BUGGY_CODE)
    open(os.path.join(d, "test_buggy.py"), "w").write(TEST_CODE)
    return d

# Create the workspace and confirm files exist
workspace = setup_workspace()
print(f"Workspace created at: {workspace}")
print("Files:", os.listdir(workspace))

### Verify the bugs — run the tests manually

Before giving the task to the agent, let's confirm the tests actually fail. This is what the agent will see when it calls `run_tests` for the first time.

In [ ]:
import subprocess
import sys

r = subprocess.run(
    [sys.executable, "test_buggy.py"],
    cwd=workspace,
    capture_output=True,
    text=True,
    timeout=15,
)
output = (r.stdout + r.stderr).strip()
print("Test output (before fix):")
print(output or "(no output)")
print(f"\nReturn code: {r.returncode}  (0 = all pass, non-zero = failures)")

## Part 4 — The Tools (ACI)

The three tools form the **Agent-Computer Interface**. Each is decorated with `@tool` from LangChain, which auto-generates an OpenAI-compatible JSON schema the LLM uses to call the tool.

### Design decisions

| Tool | Why this interface? |
|------|--------------------|
| `view_file(path)` | Returns the full file — no line-number slicing needed for small files |
| `edit_file(path, old_str, new_str)` | String replacement is safer than line-number edits (no off-by-one in the tool itself) |
| `run_tests(workspace)` | Returns raw stdout+stderr — the LLM reads Python tracebacks naturally |

### Why `@tool`?

The `@tool` decorator does two things:
1. Wraps the Python function so LangGraph can call it automatically when the LLM emits a tool call
2. Generates the JSON schema from the function signature + docstring — no manual schema writing

In [ ]:
from langchain_core.tools import tool

MAX_STEPS = 10  # safety limit — prevents infinite loops


@tool
def view_file(path: str) -> str:
    """Read and return the full contents of a file."""
    try:
        return open(path).read()
    except FileNotFoundError:
        return f"Not found: {path}"


@tool
def edit_file(path: str, old_str: str, new_str: str) -> str:
    """Replace the first occurrence of old_str with new_str in a file."""
    content = open(path).read()
    if old_str not in content:
        return f"String not found in {path}"
    open(path, "w").write(content.replace(old_str, new_str, 1))
    return "Edit applied."


@tool
def run_tests(workspace: str) -> str:
    """Run test_buggy.py in workspace and return stdout + stderr."""
    r = subprocess.run(
        ["python", "test_buggy.py"],
        cwd=workspace,
        capture_output=True,
        text=True,
        timeout=15,
    )
    return (r.stdout + r.stderr).strip() or "(no output)"


SWE_TOOLS = [view_file, edit_file, run_tests]

print("Tools registered:")
for t in SWE_TOOLS:
    print(f"  {t.name}: {t.description}")

### Inspect the auto-generated tool schema

The `@tool` decorator generates a JSON schema from the function signature and docstring. This is what the LLM sees when deciding which tool to call and with what arguments.

In [ ]:
import json

for t in SWE_TOOLS:
    print(f"=== {t.name} ===")
    print(json.dumps(t.args_schema.model_json_schema(), indent=2))
    print()

### Test the tools manually

Before wiring the tools into an agent loop, verify they work in isolation. This is good engineering practice — isolate tool failures from agent reasoning failures.

In [ ]:
buggy_path = os.path.join(workspace, "buggy.py")
test_path = os.path.join(workspace, "test_buggy.py")

# Test view_file
print("--- view_file(buggy.py) ---")
print(view_file.invoke({"path": buggy_path}))

# Test run_tests
print("\n--- run_tests(workspace) ---")
print(run_tests.invoke({"workspace": workspace}))

In [ ]:
# Test edit_file — apply a REAL fix to see it work, then restore the bug
print("--- edit_file: fix find_max ---")
result = edit_file.invoke({
    "path": buggy_path,
    "old_str": "max_val = 0  # bug: should be nums[0]",
    "new_str": "max_val = nums[0]"
})
print(result)

print("\n--- run_tests after fix ---")
print(run_tests.invoke({"workspace": workspace}))

# Restore the bug so the agent has something to fix
edit_file.invoke({
    "path": buggy_path,
    "old_str": "max_val = nums[0]",
    "new_str": "max_val = 0  # bug: should be nums[0]"
})
print("\nBug restored for agent demo.")

## Part 5 — The Workflow

### `create_react_agent` — compiled ReAct loop

`create_react_agent` from LangGraph compiles a full ReAct loop into a runnable graph in one call. Internally it builds this state machine:

```
  START
    │
    ▼
  [agent node]  ◄─────────────────────────┐
    │                                     │
    ├─ tool call? ──► [tool executor] ────┘
    │
    └─ no tool call ──► END
```

The agent node runs the LLM with the current message history. If the LLM emits a tool call, the tool executor runs it and appends the result to the message history. The loop continues until the LLM emits a plain text response.

### The system prompt

The `state_modifier` parameter injects a system prompt into every agent turn. It encodes the workflow:
1. Read the test file to understand expected behaviour
2. Read the implementation to find bugs
3. Apply one fix at a time
4. Run tests after each fix

This structured prompt is critical — it prevents the agent from applying all fixes at once (which makes debugging harder if something goes wrong).

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

SYSTEM = (
    "You are a software engineer fixing bugs in a Python codebase.\n"
    "Workflow:\n"
    "  1. view_file the test file to understand expected behaviour.\n"
    "  2. view_file the implementation to find the bugs.\n"
    "  3. edit_file to apply targeted fixes (one at a time).\n"
    "  4. run_tests to verify — repeat until all tests pass.\n"
    "Be methodical. Fix one bug at a time and verify after each edit."
)


def create_workflow():
    llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
    # create_react_agent compiles a ReAct loop: LLM -> tool call -> result -> LLM -> ...
    # It stops when the model emits a response with no tool calls (tests pass).
    return create_react_agent(llm, SWE_TOOLS, prompt=SYSTEM)


app = create_workflow()
print("Workflow compiled.")
print(f"Graph nodes: {list(app.nodes.keys())}")

### Visualise the graph (optional)

LangGraph can render its compiled state machine as a Mermaid diagram. This is useful for debugging complex multi-node graphs.

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    # Graceful fallback — requires graphviz or mermaid
    print(f"Graph render skipped ({e})")
    print("Mermaid source:")
    print(app.get_graph().draw_mermaid())

## Part 6 — Run the Agent

Now we give the agent its task. The task message mirrors exactly what `main.py` sends:
- Which workspace directory to use
- The names of the two files
- The goal: make all tests pass

We call `app.invoke()` — this is a synchronous call that blocks until the agent finishes (all tests pass or max steps reached).

In [ ]:
# Ensure bugs are present before running
# (re-create workspace in case cells above modified it)
workspace = setup_workspace()
print("Fresh workspace ready:", workspace)

task = (
    f"Workspace: {workspace}\n"
    f"Files: {workspace}/buggy.py (contains bugs) and {workspace}/test_buggy.py (failing tests).\n"
    "Fix all bugs in buggy.py so that every test in test_buggy.py passes."
)

print("\nTask sent to agent:")
print(task)

In [ ]:
print("Running agent...")
print("=" * 60)

result = app.invoke({"messages": [{"role": "user", "content": task}]})

print("Agent finished.")
print(f"Total messages in history: {len(result['messages'])}")

## Part 7 — Trace the Loop

The result contains the full message history. Each message captures one turn of the ReAct loop:
- `HumanMessage` — the task prompt
- `AIMessage` — the agent's reasoning and tool call
- `ToolMessage` — the tool's response
- Final `AIMessage` with no tool call — the agent's conclusion

Tracing this history is how you debug agent behaviour.

In [ ]:
# Pretty-print the message history — same logic as main.py
for i, msg in enumerate(result["messages"]):
    role = getattr(msg, "type", "?")
    content = str(getattr(msg, "content", ""))
    if not content:
        # AI message with only tool_calls (no text content) — show the tool call names
        tool_calls = getattr(msg, "tool_calls", [])
        if tool_calls:
            names = [tc["name"] for tc in tool_calls]
            print(f"[{i}] {role} → calls: {names}")
        continue
    if role == "ai":
        print(f"[{i}] [agent] {content[:400]}")
    elif role == "tool":
        print(f"[{i}]   [tool] {content[:200]}")
    elif role == "human":
        print(f"[{i}] [user]  {content[:100]}...")
    print()

In [ ]:
# Count tool calls by type
from collections import Counter

tool_call_counts = Counter()
for msg in result["messages"]:
    for tc in getattr(msg, "tool_calls", []):
        tool_call_counts[tc["name"]] += 1

print("Tool calls made by the agent:")
for name, count in tool_call_counts.most_common():
    print(f"  {name}: {count}")

total = sum(tool_call_counts.values())
print(f"\nTotal tool calls: {total}")

In [ ]:
# Verify the final state — read the fixed buggy.py and run tests
buggy_path = os.path.join(workspace, "buggy.py")

print("=== Fixed buggy.py ===")
print(open(buggy_path).read())

print("=== Final test run ===")
r = subprocess.run(
    ["python", "test_buggy.py"],
    cwd=workspace,
    capture_output=True,
    text=True,
    timeout=15,
)
print((r.stdout + r.stderr).strip() or "(no output)")
print(f"Return code: {r.returncode}")

### Reflect on what just happened

The agent:
1. Read the test file to understand *what correct looks like*
2. Read `buggy.py` to *locate the defects*
3. Applied fixes *one at a time* and ran tests after each
4. Continued until the test runner reported success
5. Emitted a final text response — no more tool calls needed

Notice that the agent never "knew" what the bugs were — it reasoned from test output. This is the power of the ACI pattern: the agent is grounded in executable reality, not just text.

### Key metrics to evaluate SWE-agents

| Metric | What it measures |
|--------|------------------|
| Number of tool calls | Efficiency — fewer = better reasoning |
| Number of `edit_file` calls | Precision — one per bug is ideal |
| Number of `run_tests` calls | Verification discipline |
| Final test pass rate | Correctness |
| Total tokens used | Cost |

## Part 8 — Under the Hood: How `create_react_agent` Works

Understanding the internals helps you customise and debug agents.

### The state schema

`create_react_agent` uses a `MessagesState` — a dictionary with a single `messages` key. Each LangChain message type maps to a ReAct step:

```
MessagesState["messages"] = [
  HumanMessage(content="Fix the bugs..."),  # initial task
  AIMessage(tool_calls=[...]),               # agent decides to call a tool
  ToolMessage(content="..."),                # tool result injected
  AIMessage(tool_calls=[...]),               # agent calls another tool
  ToolMessage(content="..."),                # ...
  AIMessage(content="All tests pass!"),      # agent concludes — loop exits
]
```

### The termination condition

The loop exits when the LLM emits an `AIMessage` with `tool_calls = []`. The LLM must *decide* to stop — there is no external oracle that detects test passage. This is why the system prompt says "repeat until all tests pass" — we rely on the model to recognise success from test output and stop calling tools.

In [ ]:
# Inspect the message types in the result
print("Message types in result history:")
for i, msg in enumerate(result["messages"]):
    msg_type = type(msg).__name__
    tool_calls = getattr(msg, "tool_calls", [])
    has_tc = bool(tool_calls)
    content_preview = str(getattr(msg, "content", ""))[:60].replace("\n", " ")
    print(f"  [{i:2d}] {msg_type:<20} tool_calls={has_tc}  '{content_preview}'") 

## Exercises

### Exercise 1 — Add a third bug

Add a new function `multiply_all` to `BUGGY_CODE` with a deliberate bug (e.g., it uses `+` instead of `*`). Add a corresponding test that catches the bug. Run the agent and verify it fixes the new bug too.

**Hint**: Add `def multiply_all(nums): return sum(nums)  # bug: should be product` and a test `assert multiply_all([2, 3, 4]) == 24`.

---

### Exercise 2 — Limit tool calls with `recursion_limit`

Pass `recursion_limit=5` to `app.invoke()`. What happens? Can the agent still fix both bugs? Experiment with different limits and observe the failure mode.

**Hint**: `app.invoke({...}, config={"recursion_limit": 5})`

---

### Exercise 3 — Stream intermediate steps

Replace `app.invoke()` with `app.stream()` and print each chunk as it arrives. This lets you watch the agent reason in real time.

**Hint**: `for chunk in app.stream({"messages": [{...}]}): print(chunk)`

In [ ]:
# --- ANSWER KEY ---

# Exercise 1: Add multiply_all bug
BUGGY_CODE_EX1 = """\
def find_max(nums):
    max_val = 0  # bug: should be nums[0]
    for n in nums:
        if n > max_val:
            max_val = n
    return max_val

def average(nums):
    return sum(nums) / len(nums) - 1  # bug: spurious -1

def multiply_all(nums):
    return sum(nums)  # bug: should be product, not sum
"""

TEST_CODE_EX1 = """\
from buggy import find_max, average, multiply_all

def test_find_max():
    assert find_max([3, 1, 4, 1, 5, 9]) == 9
    assert find_max([-1, -2, -3]) == -1

def test_average():
    assert average([1, 2, 3, 4, 5]) == 3.0
    assert average([10, 20]) == 15.0

def test_multiply_all():
    assert multiply_all([2, 3, 4]) == 24   # sum would give 9
    assert multiply_all([1, 5]) == 5

if __name__ == "__main__":
    test_find_max()
    test_average()
    test_multiply_all()
    print("All tests passed!")
"""

# Set up exercise 1 workspace
ws_ex1 = tempfile.mkdtemp(prefix="swe_ex1_")
open(os.path.join(ws_ex1, "buggy.py"), "w").write(BUGGY_CODE_EX1)
open(os.path.join(ws_ex1, "test_buggy.py"), "w").write(TEST_CODE_EX1)
print(f"Exercise 1 workspace: {ws_ex1}")

task_ex1 = (
    f"Workspace: {ws_ex1}\n"
    f"Files: {ws_ex1}/buggy.py (contains 3 bugs) and {ws_ex1}/test_buggy.py (failing tests).\n"
    "Fix all bugs in buggy.py so that every test in test_buggy.py passes."
)

print("Running exercise 1 agent...")
result_ex1 = app.invoke({"messages": [{"role": "user", "content": task_ex1}]})

# Verify
r_ex1 = subprocess.run(
    ["python", "test_buggy.py"],
    cwd=ws_ex1,
    capture_output=True, text=True, timeout=15
)
print("Exercise 1 result:")
print((r_ex1.stdout + r_ex1.stderr).strip())

In [ ]:
# Exercise 2: recursion_limit
ws_ex2 = setup_workspace()  # fresh workspace with 2 bugs
task_ex2 = (
    f"Workspace: {ws_ex2}\n"
    f"Files: {ws_ex2}/buggy.py (contains bugs) and {ws_ex2}/test_buggy.py (failing tests).\n"
    "Fix all bugs in buggy.py so that every test in test_buggy.py passes."
)

print("Running with recursion_limit=5...")
try:
    result_ex2 = app.invoke(
        {"messages": [{"role": "user", "content": task_ex2}]},
        config={"recursion_limit": 5}
    )
    print(f"Completed with {len(result_ex2['messages'])} messages.")
except Exception as e:
    print(f"Hit recursion limit: {type(e).__name__}: {e}")
    print("\n=> This is the expected failure mode when the agent can't finish in N steps.")

In [ ]:
# Exercise 3: stream intermediate steps
ws_ex3 = setup_workspace()
task_ex3 = (
    f"Workspace: {ws_ex3}\n"
    f"Files: {ws_ex3}/buggy.py (contains bugs) and {ws_ex3}/test_buggy.py (failing tests).\n"
    "Fix all bugs in buggy.py so that every test in test_buggy.py passes."
)

print("Streaming agent steps...")
print("=" * 60)

for chunk in app.stream({"messages": [{"role": "user", "content": task_ex3}]}):
    # Each chunk is a dict of {node_name: state_update}
    for node_name, state_update in chunk.items():
        msgs = state_update.get("messages", [])
        for msg in msgs:
            role = getattr(msg, "type", "?")
            content = str(getattr(msg, "content", ""))[:200]
            tool_calls = getattr(msg, "tool_calls", [])
            if tool_calls:
                names = [tc["name"] for tc in tool_calls]
                print(f"[{node_name}] {role} → calls {names}")
            elif content:
                print(f"[{node_name}] {role}: {content}")

print("=" * 60)
print("Stream complete.")

---

## Workshop Complete

You've built a working SWE-agent from scratch. You understand:
- The **ACI pattern** — minimal, lossless, actionable tool design
- How `@tool` generates JSON schemas the LLM uses for structured calls
- How `create_react_agent` compiles a ReAct loop without boilerplate
- How to trace, inspect, and debug the message history
- How to customise the loop (streaming, recursion limits)

### Going further

| Topic | Direction |
|-------|-----------|
| Larger codebases | Add `search_files` and `run_linter` tools to the ACI |
| Real GitHub issues | Point the agent at actual repos using `gh` CLI tools |
| Multi-file bugs | Give `edit_file` a line-range mode for targeted edits in large files |
| Evaluation | Benchmark on SWE-bench Lite — 300 real GitHub issues |

---

**Next: Example 102 — [check the examples directory for the next topic]**

> Yang et al. 2024 — *SWE-agent: Agent-Computer Interfaces Enable Automated Software Engineering* — [arxiv.org/abs/2405.15793](https://arxiv.org/abs/2405.15793)